<a href="https://colab.research.google.com/github/Srishti160/Oral-Cancer-Detection/blob/main/oral_cancer_detection_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# First row: Raman shift values for each Var column
raman_shifts = df.iloc[0, 2:].astype(float).values  # from Var0 to Var1037

# Remaining rows: actual samples
df_cleaned = df[df['raman_type'] != 'raman_shift'].copy()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# Load dataset
df = pd.read_csv("/oral_cancer.csv")

# Remove the Raman shift row
raman_shifts = df.iloc[0, 2:].astype(float).values
df_samples = df[df['raman_type'] != 'raman_shift'].copy()

# Convert labels to binary: 0 (Healthy), 1 (Cancerous)
df_samples['labels'] = pd.to_numeric(df_samples['labels'], errors='coerce')
df_samples.dropna(subset=['labels'], inplace=True)
df_samples['labels'] = df_samples['labels'].astype(int)
df_samples['binary_labels'] = df_samples['labels'].apply(lambda x: 0 if x == 0 else 1)

# Extract features and binary labels
X = df_samples.iloc[:, 2:].astype(float).values
y = df_samples['binary_labels'].values

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

# Define models
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "SVM": SVC(probability=True, class_weight='balanced'),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

# Train and evaluate
for name, model in models.items():
    print(f"\n🔍 {name}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["Healthy", "Cancerous"]))



🔍 Logistic Regression
              precision    recall  f1-score   support

     Healthy       1.00      1.00      1.00       141
   Cancerous       1.00      1.00      1.00       566

    accuracy                           1.00       707
   macro avg       1.00      1.00      1.00       707
weighted avg       1.00      1.00      1.00       707


🔍 Random Forest
              precision    recall  f1-score   support

     Healthy       0.98      0.91      0.94       141
   Cancerous       0.98      0.99      0.99       566

    accuracy                           0.98       707
   macro avg       0.98      0.95      0.96       707
weighted avg       0.98      0.98      0.98       707


🔍 SVM
              precision    recall  f1-score   support

     Healthy       0.99      0.97      0.98       141
   Cancerous       0.99      1.00      1.00       566

    accuracy                           0.99       707
   macro avg       0.99      0.98      0.99       707
weighted avg       0.99    

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:06:50] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

     Healthy       1.00      1.00      1.00       141
   Cancerous       1.00      1.00      1.00       566

    accuracy                           1.00       707
   macro avg       1.00      1.00      1.00       707
weighted avg       1.00      1.00      1.00       707



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# STEP 1: Load and preprocess data
df = pd.read_csv("/content/oral_cancer.csv")  # Replace with your file path

# Remove meta row ("raman_shift") and clean labels
df_samples = df[df['raman_type'] != 'raman_shift'].copy()
df_samples['labels'] = pd.to_numeric(df_samples['labels'], errors='coerce')
df_samples.dropna(subset=['labels'], inplace=True)
df_samples['labels'] = df_samples['labels'].astype(int)

# Convert to binary classification
df_samples['binary_labels'] = df_samples['labels'].apply(lambda x: 0 if x == 0 else 1)

# Extract features and labels
X = df_samples.iloc[:, 2:].astype(float).values
y = df_samples['binary_labels'].values

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# STEP 2: Define XGBoost model
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# STEP 3: Define scoring metrics
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score),
    'roc_auc': make_scorer(roc_auc_score)
}

# STEP 4: Perform 5-fold cross-validation
results = cross_validate(model, X_scaled, y, cv=5, scoring=scoring, return_train_score=False)

# STEP 5: Print average and std for each metric
print("📊 Cross-Validation Results (5-fold):")
for metric in scoring.keys():
    scores = results[f'test_{metric}']
    print(f"\n🔍 {metric.upper()}")
    print(f"Scores: {np.round(scores, 4)}")
    print(f"Mean: {np.mean(scores):.4f}")
    print(f"Std Dev: {np.std(scores):.4f}")

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [17:39:43] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [17:39:46] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [17:39:49] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [17:39:51] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [17:39:54] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

📊 Cross-Validation Results (5-fold):

🔍 ACCURACY
Scores: [1. 1. 1. 1. 1.]
Mean: 1.0000
Std Dev: 0.0000

🔍 PRECISION
Scores: [1. 1. 1. 1. 1.]
Mean: 1.0000
Std Dev: 0.0000

🔍 RECALL
Scores: [1. 1. 1. 1. 1.]
Mean: 1.0000
Std Dev: 0.0000

🔍 F1
Scores: [1. 1. 1. 1. 1.]
Mean: 1.0000
Std Dev: 0.0000

🔍 ROC_AUC
Scores: [1. 1. 1. 1. 1.]
Mean: 1.0000
Std Dev: 0.0000


In [ ]:
# Ensure Raman shift row is removed before training
df = pd.read_csv("oral_cancer.csv")

# Remove the meta row
df_samples = df[df['raman_type'] != 'raman_shift'].copy()

# Sanity check: should show values like 'TSCC' or 'Normal'
print("Unique raman_type values after cleaning:", df_samples['raman_type'].unique())


Unique raman_type values after cleaning: ['Health' 'Benign' 'High-T1-N0M0' 'High-T2-N0M0' 'High-T2-N1M0'
 'High-T3-N1M0' 'High-T4-N0M0' 'High-T4-N1M0' 'High-T4-N2M0'
 'High-Tis-N0M0' 'Low-T4-N0M0' 'Low-T4-N2M0' 'Medium-T1-N0M0'
 'Medium-T2-N0M0' 'Medium-T2-N1M0' 'Medium-T3-N0M0' 'Medium-T4-N0M0'
 'Medium-T4-N2M0']


In [ ]:
# This should only include numeric Raman intensities — no labels, no raman_type
print("Training features preview (X):")
print(df_samples.iloc[:, 2:6].head())

# This should include only 0 (Healthy) and 1 (Cancerous)
print("Binary label preview (y):")
print(df_samples['binary_labels'].value_counts())


Training features preview (X):
        Var0       Var1       Var2       Var3
1  63103.476  63104.476  63104.476  63102.476
2  63103.648  63104.648  63104.648  63102.648
3  63103.750  63104.750  63104.750  63102.750
4  63102.350  63103.350  63103.350  63101.350
5  63102.876  63103.876  63103.876  63101.876
Binary label preview (y):


KeyError: 'binary_labels'